# UAV GPS Localization — Real-World Test Suite
## ICMTC 2026 · Dummy Data → Drop-In Real Data When Ready

Simulates a realistic flight with dummy data structured exactly like a real GoPro + ArduPilot flight.
When you have real hardware, only the **loader cells** change. All test logic and scoring stay the same.

| Dummy (now) | Real (later) | Section |
|---|---|---|
| `load_camera_params("dummy")` | Load `camera_params.yaml` | 1 |
| `simulate_flight()` | Parse `.bin` / `.tlog` telemetry | 2 |
| `simulate_detections()` | Run YOLO on video frames | 3 |
| Hard-coded GPS coordinates | Surveyed-points spreadsheet | 5 |


## Section 1 — Camera Parameters

In [ ]:
import numpy as np, math, random
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

SEED = 42
random.seed(SEED); np.random.seed(SEED)

@dataclass
class CameraIntrinsics:
    """Pinhole camera model. Replace numbers with your calibration output."""
    fx: float = 1452.7   # GoPro Hero 10/11 Linear @ 1080p — realistic placeholder
    fy: float = 1453.1
    cx: float =  963.2
    cy: float =  541.8
    dist: np.ndarray = field(
        default_factory=lambda: np.array([-0.022, 0.008, 0.0003, -0.0001, 0.0]))
    # k1 < 0 = mild barrel distortion typical of GoPro Linear mode

    @property
    def K(self) -> np.ndarray:
        return np.array([[self.fx, 0, self.cx],
                         [0, self.fy, self.cy],
                         [0,      0,       1]], dtype=float)

    def fov_half_m(self, altitude_m: float) -> Tuple[float, float]:
        """Half-width/height of the visible ground area (metres) at given altitude."""
        return (self.cx / self.fx) * altitude_m, (self.cy / self.fy) * altitude_m


def load_camera_params(source: str = "dummy") -> CameraIntrinsics:
    """
    ── SWAP-IN FOR REAL DATA ──────────────────────────────────────────────
    import yaml
    with open('camera_params.yaml') as f:
        p = yaml.safe_load(f)
    return CameraIntrinsics(fx=p['fx'], fy=p['fy'], cx=p['cx'], cy=p['cy'],
                             dist=np.array(p['dist_coeffs']))
    ──────────────────────────────────────────────────────────────────────
    """
    return CameraIntrinsics()   # returns realistic placeholder


CAM = load_camera_params()

# ── Geodesy & mount ────────────────────────────────────────────────────────
METERS_PER_DEG_LAT = 111_320.0
LAT_REF             = 30.05          # Cairo airport
METERS_PER_DEG_LON  = 111_320.0 * math.cos(math.radians(LAT_REF))

# Camera mount: cam+X→body+Y(East), cam+Y→body-X(South=−North), cam+Z→body+Z(Down)
R_MOUNT = np.array([[ 0, -1, 0],
                    [ 1,  0, 0],
                    [ 0,  0, 1]], dtype=float)

hw, hh = CAM.fov_half_m(80)
print(f"✅ Camera loaded  fx={CAM.fx}  fy={CAM.fy}  cx={CAM.cx}  cy={CAM.cy}")
print(f"   Ground FOV at 80 m: ±{hw:.1f} m E/W  ×  ±{hh:.1f} m N/S")
print(f"   A 1×2 m flag at 80 m spans {CAM.fx/80:.0f}×{2*CAM.fy/80:.0f} px")


## Section 2 — Simulated Flight Telemetry

The UAV starts **upstream** of the flag and flies over it, so the flag passes through the camera FOV.
Each `TelemetryFrame` matches the field names from an ArduPilot `.bin` log.


In [ ]:
@dataclass
class TelemetryFrame:
    """One telemetry sample — mirrors ArduPilot GPS+ATT+BARO message fields."""
    time_s:    float
    lat_uav:   float   # degrees
    lon_uav:   float   # degrees
    alt_agl:   float   # metres AGL (barometer, zeroed at takeoff)
    roll_deg:  float
    pitch_deg: float
    yaw_deg:   float   # true heading (magnetic + declination correction applied)


def simulate_flight(
    flag_lat: float, flag_lon: float,  # UAV flies OVER this point
    altitude_m:  float = 80.0,
    heading_deg: float = 0.0,
    speed_m_s:   float = 18.0,
    pass_duration_s: float = 20.0,    # total pass duration; flag is at midpoint
    fps: float = 30.0,
    gps_noise_m: float = 3.0,         # GPS CEP (metres)
    alt_noise_m: float = 1.5,         # barometer noise
    att_noise_deg: float = 1.0,       # IMU noise
    wind_pitch_deg: float = 2.0,      # systematic nose-up bias (wind resistance)
    seed: int = SEED,
) -> List[TelemetryFrame]:
    """
    Straight-line pass: flag at midpoint of trajectory.

    ── SWAP-IN FOR REAL DATA ──────────────────────────────────────────────
    from pymavlink import mavutil
    mlog = mavutil.mavlink_connection('flight.bin')
    frames = []
    while True:
        msg = mlog.recv_match(blocking=False)
        if msg is None: break
        if msg.get_type() == 'GPS':
            ...
    return frames
    ──────────────────────────────────────────────────────────────────────
    """
    rng = np.random.default_rng(seed)
    n   = int(pass_duration_s * fps)
    half_dist = speed_m_s * pass_duration_s / 2   # metres before flag

    # Origin: flag − half_dist along heading
    origin_lat = flag_lat - half_dist * math.cos(math.radians(heading_deg)) / METERS_PER_DEG_LAT
    origin_lon = flag_lon - half_dist * math.sin(math.radians(heading_deg)) / METERS_PER_DEG_LON

    frames = []
    for i in range(n):
        t = i / fps
        dist = speed_m_s * t
        true_lat = origin_lat + dist * math.cos(math.radians(heading_deg)) / METERS_PER_DEG_LAT
        true_lon = origin_lon + dist * math.sin(math.radians(heading_deg)) / METERS_PER_DEG_LON

        lat = true_lat + rng.normal(0, gps_noise_m / METERS_PER_DEG_LAT)
        lon = true_lon + rng.normal(0, gps_noise_m / METERS_PER_DEG_LON)
        alt = altitude_m + rng.normal(0, alt_noise_m) + 0.5 * math.sin(t * 0.3)

        frames.append(TelemetryFrame(
            time_s    = t,
            lat_uav   = lat,
            lon_uav   = lon,
            alt_agl   = alt,
            roll_deg  = rng.normal(0, att_noise_deg),
            pitch_deg = wind_pitch_deg + rng.normal(0, att_noise_deg),
            yaw_deg   = heading_deg    + rng.normal(0, att_noise_deg),
        ))
    return frames


FLAG_LAT = LAT_REF + 200 / METERS_PER_DEG_LAT
FLAG_LON = 31.25   + 200 / METERS_PER_DEG_LON

test_flight = simulate_flight(FLAG_LAT, FLAG_LON, heading_deg=45, altitude_m=80)
print(f"✅ Simulated flight: {len(test_flight)} frames over {len(test_flight)/30:.1f} s")
print(f"   Alt  mean ± σ : {np.mean([f.alt_agl   for f in test_flight]):.1f} ± "
      f"{np.std([f.alt_agl   for f in test_flight]):.1f} m")
print(f"   Pitch mean    : {np.mean([f.pitch_deg for f in test_flight]):.1f}°  (wind-induced)")


## Section 3 — Simulated YOLO Detections

`flag_to_pixel()` is the **inverse** of `localize()` — given the true flag GPS and telemetry, it computes
the exact pixel YOLO should detect. We then add realistic bbox jitter and missed-detection probability.


In [ ]:
def rx(a): c,s=math.cos(a),math.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]])
def ry(a): c,s=math.cos(a),math.sin(a); return np.array([[c,0,s],[0,1,0],[-s,0,c]])
def rz(a): c,s=math.cos(a),math.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]])

def build_world_rotation(roll_deg, pitch_deg, yaw_deg, Rm=R_MOUNT):
    r,p,y = math.radians(roll_deg), math.radians(pitch_deg), math.radians(yaw_deg)
    return rz(y) @ ry(p) @ rx(r) @ Rm


def flag_to_pixel(flag_lat: float, flag_lon: float,
                  tel: TelemetryFrame,
                  cam: CameraIntrinsics) -> Optional[Tuple[float, float]]:
    """
    Compute the ideal pixel of a flag given its GPS and current telemetry.
    Returns None if the flag is outside the camera frame.
    This is the INVERSE of localize() — used only for test data generation.
    """
    N = (flag_lat - tel.lat_uav) * METERS_PER_DEG_LAT
    E = (flag_lon - tel.lon_uav) * METERS_PER_DEG_LON
    D = tel.alt_agl
    r_ned = np.array([N, E, D])

    if D <= 0:
        return None   # flag above UAV — impossible for ground target

    R = build_world_rotation(tel.roll_deg, tel.pitch_deg, tel.yaw_deg)
    r_cam = R.T @ r_ned   # inverse: NED → camera frame

    if r_cam[2] <= 0:
        return None   # flag behind the camera

    # Project through pinhole
    u = (r_cam[0] / r_cam[2]) * cam.fx + cam.cx
    v = (r_cam[1] / r_cam[2]) * cam.fy + cam.cy

    if not (0 <= u < 1920 and 0 <= v < 1080):
        return None   # flag outside frame

    return float(u), float(v)


def simulate_detections(
    flag_lat: float, flag_lon: float,
    flight: List[TelemetryFrame],
    cam: CameraIntrinsics,
    bbox_noise_px:  float = 3.0,   # YOLO bbox centre jitter (pixels)
    detection_rate: float = 0.8,   # probability YOLO fires on a frame where flag is visible
    conf_noise:     float = 0.15,
    seed: int = SEED + 1,
) -> List[dict]:
    """
    Simulate YOLO detections across a flight.

    ── SWAP-IN FOR REAL DATA ──────────────────────────────────────────────
    from ultralytics import YOLO
    model = YOLO('flag_detector.pt')
    dets  = []
    cap   = cv2.VideoCapture('video.MP4')
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % 5 == 0:   # sample every 5th frame
            for box in model(frame, verbose=False)[0].boxes:
                t_sec = frame_idx / cap.get(cv2.CAP_PROP_FPS)
                tel   = min(telemetry, key=lambda f: abs(f.time_s - t_sec))
                dets.append({'u':(box.x1+box.x2)/2, 'v':(box.y1+box.y2)/2,
                             'confidence': float(box.conf),
                             'altitude': tel.alt_agl, 'lat_uav': tel.lat_uav,
                             'lon_uav': tel.lon_uav,  'roll': tel.roll_deg,
                             'pitch': tel.pitch_deg,  'yaw': tel.yaw_deg})
        frame_idx += 1
    return dets
    ──────────────────────────────────────────────────────────────────────
    """
    rng  = np.random.default_rng(seed)
    dets = []
    for i, tel in enumerate(flight):
        px = flag_to_pixel(flag_lat, flag_lon, tel, cam)
        if px is None:
            continue
        if rng.random() > detection_rate:
            continue   # YOLO missed this frame
        u, v = px
        u += rng.normal(0, bbox_noise_px)
        v += rng.normal(0, bbox_noise_px)
        conf = float(np.clip(rng.normal(0.75, conf_noise), 0.1, 1.0))
        dets.append({
            'u':         u,     'v':         v,     'confidence': conf,
            'frame_idx': i,     'time_s':    tel.time_s,
            'altitude':  tel.alt_agl,
            'lat_uav':   tel.lat_uav,  'lon_uav':   tel.lon_uav,
            'roll':      tel.roll_deg, 'pitch':      tel.pitch_deg,
            'yaw':       tel.yaw_deg,
        })
    return dets


dets_test = simulate_detections(FLAG_LAT, FLAG_LON, test_flight, CAM)
print(f"✅ Simulated {len(dets_test)} detections from {len(test_flight)} frames")
print(f"   Detection rate : {len(dets_test)/len(test_flight)*100:.0f}%")
if dets_test:
    print(f"   Confidence mean: {np.mean([d['confidence'] for d in dets_test]):.2f}")
    print(f"   Pixel u range  : {min(d['u'] for d in dets_test):.0f}–{max(d['u'] for d in dets_test):.0f}")
    print(f"   Pixel v range  : {min(d['v'] for d in dets_test):.0f}–{max(d['v'] for d in dets_test):.0f}")


## Section 4 — Core Pipeline (self-contained copy)

In [ ]:
def undistort_pixel(u, v, K, dist):
    """Brown-Conrady single-pixel undistortion (5 Newton iterations)."""
    fx,fy,cx,cy = K[0,0],K[1,1],K[0,2],K[1,2]
    k1,k2,p1,p2 = dist[0],dist[1],dist[2],dist[3]
    k3 = dist[4] if len(dist) > 4 else 0.
    xd,yd = (u-cx)/fx, (v-cy)/fy
    x,y = xd,yd
    for _ in range(5):
        r2  = x*x + y*y
        rad = 1 + k1*r2 + k2*r2**2 + k3*r2**3
        dx  = 2*p1*x*y + p2*(r2 + 2*x*x)
        dy  = p1*(r2 + 2*y*y) + 2*p2*x*y
        x = (xd-dx)/rad; y = (yd-dy)/rad
    return x*fx+cx, y*fy+cy


def localize(u_box, v_box, cam, altitude_m, lat_uav, lon_uav,
             roll_deg=0., pitch_deg=0., yaw_deg=0., Rm=R_MOUNT, min_rz=0.17):
    """Full pipeline: pixel → GPS. Returns None if geometry invalid."""
    u_u, v_u = undistort_pixel(u_box, v_box, cam.K, cam.dist)
    x_n = (u_u - cam.cx) / cam.fx
    y_n = (v_u - cam.cy) / cam.fy
    r_cam = np.array([x_n, y_n, 1.])
    R = build_world_rotation(roll_deg, pitch_deg, yaw_deg, Rm)
    r_ned = R @ r_cam
    if r_ned[2] < min_rz: return None
    t = altitude_m / r_ned[2]
    N, E = t * r_ned[0], t * r_ned[1]
    return lat_uav + N/METERS_PER_DEG_LAT, lon_uav + E/METERS_PER_DEG_LON


def localize_flag(dets, cam, min_confidence=0.4):
    """Median aggregate of per-frame estimates."""
    est = []
    for d in dets:
        if d.get('confidence', 1.) < min_confidence: continue
        r = localize(d['u'], d['v'], cam, d['altitude'],
                     d['lat_uav'], d['lon_uav'],
                     d.get('roll',0.), d.get('pitch',0.), d.get('yaw',0.))
        if r: est.append(r)
    if not est: return None
    a = np.array(est)
    return float(np.median(a[:,0])), float(np.median(a[:,1]))


def haversine_m(la1,lo1,la2,lo2):
    R=6_371_000.
    p1,p2=math.radians(la1),math.radians(la2)
    dp,dl=math.radians(la2-la1),math.radians(lo2-lo1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.asin(math.sqrt(a))


def score(el,elo,tl,tlo):
    d=haversine_m(el,elo,tl,tlo)
    p=15 if d<=20 else(12 if d<=30 else(9 if d<=40 else(6 if d<=50 else(3 if d<=60 else 0))))
    return p,d


# ── Quick smoke test ──────────────────────────────────────────────────────
result = localize_flag(dets_test, CAM)
if result:
    pts, err = score(*result, FLAG_LAT, FLAG_LON)
    print(f"✅ Smoke test: error = {err:.2f} m  →  {pts} pts  "
          f"({'✅ Full points!' if pts==15 else '🟨 Partial'})")
else:
    print("❌ Smoke test: no valid localization")


## Section 5 — Scenario Tests with Known Ground Truth

Nine scenarios escalating from ideal to adversarial.  
Each reports error in metres and competition points.


In [ ]:
BASE_LAT = LAT_REF + 300 / METERS_PER_DEG_LAT
BASE_LON = 31.25   + 300 / METERS_PER_DEG_LON

def run_scenario(name, flat, flon, heading_deg, altitude_m,
                 wind_pitch_deg=2., gps_noise_m=3.,
                 use_imu=True, n_frames=10, extra=""):
    flight = simulate_flight(flat, flon, altitude_m=altitude_m,
                              heading_deg=heading_deg, speed_m_s=18.,
                              pass_duration_s=20., fps=30.,
                              gps_noise_m=gps_noise_m,
                              wind_pitch_deg=wind_pitch_deg)
    dets = simulate_detections(flat, flon, flight, CAM)

    if not dets:
        return {'name':name,'error_m':None,'pts':0,'n_dets':0,'extra':extra}

    if not use_imu:
        dets = [dict(d, roll=0., pitch=0.) for d in dets]

    subset = dets[:n_frames]
    result = localize_flag(subset, CAM)
    if result is None:
        return {'name':name,'error_m':None,'pts':0,'n_dets':len(subset),'extra':extra}

    pts, err = score(*result, flat, flon)
    return {'name':name,'error_m':err,'pts':pts,'n_dets':len(subset),'extra':extra,
            'est':result,'true':(flat,flon)}


SCENARIOS = [
    run_scenario("1. Ideal — calm, accurate GPS",   BASE_LAT,BASE_LON, 0,  80, 0,  1,  True, 15),
    run_scenario("2. Nominal — heading NE, wind",   BASE_LAT,BASE_LON, 45, 80, 2,  3,  True, 10),
    run_scenario("3. Heading East",                 BASE_LAT,BASE_LON, 90, 80, 2,  3,  True, 10),
    run_scenario("4. Heading South-West",           BASE_LAT,BASE_LON,225, 80, 2,  3,  True, 10),
    run_scenario("5. High altitude (100 m)",        BASE_LAT,BASE_LON, 45,100, 2,  3,  True, 10),
    run_scenario("6. Low altitude (50 m)",          BASE_LAT,BASE_LON, 45, 50, 2,  3,  True, 10),
    run_scenario("7. Single frame only",            BASE_LAT,BASE_LON, 45, 80, 2,  3,  True,  1),
    run_scenario("8. No IMU — pitch uncompensated", BASE_LAT,BASE_LON, 45, 80, 5,  3, False, 10,
                 extra="5° pitch, no IMU correction"),
    run_scenario("9. Worst case — bad GPS+wind",    BASE_LAT,BASE_LON, 45, 80, 5,  6,  True, 10),
]

BIN = {15:"✅ 15pts",12:"🟩 12pts",9:"🟨  9pts",6:"🟧  6pts",3:"🟥  3pts",0:"❌  0pts"}
print(f"{'Scenario':<42} {'Error':>8}  {'Dets':>5}  Result")
print("─"*75)
for s in SCENARIOS:
    e = s['error_m']
    tag = BIN[s['pts']]
    extra = f"  ← {s['extra']}" if s['extra'] else ""
    if e is None:
        print(f"  {s['name']:<40} {'N/A':>8}  {s['n_dets']:>5}  ❌ no detections{extra}")
    else:
        print(f"  {s['name']:<40} {e:>7.2f}m  {s['n_dets']:>5}  {tag}{extra}")


## Section 6 — Full Mission Simulation (5 flags, 3 passes)

In [ ]:
FLAGS = {
    'Flag A': (LAT_REF + 150/METERS_PER_DEG_LAT, 31.25 + 100/METERS_PER_DEG_LON),
    'Flag B': (LAT_REF + 350/METERS_PER_DEG_LAT, 31.25 + 280/METERS_PER_DEG_LON),
    'Flag C': (LAT_REF + 220/METERS_PER_DEG_LAT, 31.25 -  60/METERS_PER_DEG_LON),
    'Flag D': (LAT_REF +  90/METERS_PER_DEG_LAT, 31.25 + 420/METERS_PER_DEG_LON),
    'Flag E': (LAT_REF + 420/METERS_PER_DEG_LAT, 31.25 +  40/METERS_PER_DEG_LON),
}

PASS_HEADINGS = [45, 90, 0]   # 3 passes at different headings

all_dets: Dict[str, List[dict]] = {name: [] for name in FLAGS}

for hdg in PASS_HEADINGS:
    for fname, (flat, flon) in FLAGS.items():
        flight = simulate_flight(flat, flon, altitude_m=80, heading_deg=hdg,
                                  speed_m_s=18, pass_duration_s=20, fps=30,
                                  gps_noise_m=3, wind_pitch_deg=2,
                                  seed=SEED + PASS_HEADINGS.index(hdg) * 10)
        dets = simulate_detections(flat, flon, flight, CAM,
                                    seed=SEED + 100 + PASS_HEADINGS.index(hdg) * 10)
        all_dets[fname].extend(dets)

total_pts = 0
print("FULL MISSION SCORING")
print("="*65)
for fname, (flat, flon) in FLAGS.items():
    dets = all_dets[fname]
    if not dets:
        print(f"  {fname}: ❌ no detections"); continue
    result = localize_flag(dets, CAM)
    if result is None:
        print(f"  {fname}: ❌ localization failed"); continue
    pts, err = score(*result, flat, flon)
    total_pts += pts
    icon = BIN[pts]
    print(f"  {fname}: {icon}   error={err:6.2f} m   n={len(dets)} detections")

print("─"*65)
max_pts = len(FLAGS) * 15
print(f"  TOTAL: {total_pts} / {max_pts} pts  ({total_pts/max_pts*100:.0f}% of flag-location score)")


## Section 7 — Error Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("UAV GPS Localisation — Synthetic Test Results", fontsize=13, fontweight='bold')

# ── Plot 1: Error vs N frames ─────────────────────────────────────────
ax = axes[0]
ax.set_title("Error vs Frames Averaged (Flag A, heading 45°)")
fl_a = list(FLAGS.values())[0]
flt_a = simulate_flight(*fl_a, altitude_m=80, heading_deg=45, pass_duration_s=25, fps=30, gps_noise_m=3)
det_a = simulate_detections(*fl_a, flt_a, CAM)

ns, errors = [], []
for n in range(1, min(len(det_a)+1, 31)):
    r = localize_flag(det_a[:n], CAM)
    ns.append(n)
    errors.append(haversine_m(*r, *fl_a) if r else None)

v_n = [n for n,e in zip(ns,errors) if e is not None]
v_e = [e for e in errors if e is not None]
ax.plot(v_n, v_e, 'b-o', ms=4, lw=1.5)
ax.axhline(20, color='green', lw=1.5, ls='--', label='20 m (full pts)')
ax.axhline(30, color='orange', lw=1.5, ls='--', label='30 m (12 pts)')
ax.set_xlabel("Frames used"); ax.set_ylabel("Error (m)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_ylim(0, max(v_e)*1.4 if v_e else 50)

# ── Plot 2: Scenario bar chart ────────────────────────────────────────
ax = axes[1]
ax.set_title("Error by Scenario")
sc_names = [s['name'].split('—')[0].strip() for s in SCENARIOS if s['error_m'] is not None]
sc_errs  = [s['error_m'] for s in SCENARIOS if s['error_m'] is not None]
colours  = ['#2ecc71' if e<=20 else ('#f39c12' if e<=30 else '#e74c3c') for e in sc_errs]
bars = ax.barh(sc_names, sc_errs, color=colours, edgecolor='white', height=0.6)
ax.axvline(20, color='green', lw=1.5, ls='--')
ax.axvline(30, color='orange', lw=1.5, ls='--')
ax.set_xlabel("Error (m)")
for bar, e in zip(bars, sc_errs):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{e:.1f}m', va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3)

# ── Plot 3: Ground map of estimates ──────────────────────────────────
ax = axes[2]
ax.set_title("Flag A — Per-Frame vs Aggregated")
per_frame = []
for d in det_a[:25]:
    r = localize(d['u'],d['v'],CAM,d['altitude'],d['lat_uav'],d['lon_uav'],
                 d['roll'],d['pitch'],d['yaw'])
    if r: per_frame.append(r)

if per_frame:
    flat, flon = fl_a
    pf = np.array(per_frame)
    dx = (pf[:,1]-flon)*METERS_PER_DEG_LON
    dy = (pf[:,0]-flat)*METERS_PER_DEG_LAT

    agg = localize_flag(det_a[:25], CAM)
    adx = (agg[1]-flon)*METERS_PER_DEG_LON if agg else 0
    ady = (agg[0]-flat)*METERS_PER_DEG_LAT if agg else 0

    ax.scatter(dx,dy,s=30,c='steelblue',alpha=0.6,label='Single-frame')
    ax.scatter([0],[0],s=150,c='red',marker='*',zorder=5,label='True flag')
    if agg:
        ax.scatter([adx],[ady],s=120,c='limegreen',marker='D',zorder=5,label='Aggregated')
    for r_m, col, lbl in [(20,'green','20m'),(30,'orange','30m')]:
        ax.add_patch(plt.Circle((0,0),r_m,color=col,fill=False,ls='--',lw=1.3))
        ax.text(r_m*0.68,r_m*0.68,lbl,color=col,fontsize=8)
    lim = max(35, max(abs(dx).max(), abs(dy).max())*1.2)
    ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim)
    ax.set_xlabel("East offset (m)"); ax.set_ylabel("North offset (m)")
    ax.legend(fontsize=8); ax.set_aspect('equal'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/uav_results.png', dpi=130, bbox_inches='tight')
plt.show()
print("✅ Plots rendered.")


## Section 8 — Stress Tests & Edge Cases

Deliberate adversarial conditions with clear pass/fail limits.

In [ ]:
PASS_ALL = True

def st(label, err, limit):
    global PASS_ALL
    if err is None:
        print(f"  ❌ {label}: pipeline returned None"); PASS_ALL=False; return
    ok = err <= limit
    if not ok: PASS_ALL = False
    print(f"  {'✅' if ok else '❌'} {label}: {err:.2f} m  (limit = {limit} m)")


print("ST-1: All cardinal headings (ideal conditions)")
for hd, hl in [(0,"N"),(90,"E"),(180,"S"),(270,"W")]:
    s = run_scenario(f"  {hl}", BASE_LAT,BASE_LON, hd, 80, 0, 1, True, 15)
    st(f"Heading {hl}", s['error_m'], 8)

print("\nST-2: IMU compensation removes pitch bias")
s_y = run_scenario("imu",   BASE_LAT,BASE_LON,45,80,5,1,True,  15)
s_n = run_scenario("noimu", BASE_LAT,BASE_LON,45,80,5,1,False, 15)
st("5° pitch WITH IMU",    s_y['error_m'], 15)
# Without IMU should have MORE error — if it doesn't, our simulation or pipeline has a bug
if s_n['error_m'] and s_y['error_m']:
    ok = s_n['error_m'] > s_y['error_m']
    if not ok: PASS_ALL = False
    print(f"  {'✅' if ok else '❌'} Without IMU ({s_n['error_m']:.2f}m) > With IMU ({s_y['error_m']:.2f}m)")

print("\nST-3: Altitude range 50–100 m")
for alt in [50, 70, 80, 100]:
    s = run_scenario(f"{alt}m", BASE_LAT,BASE_LON,45,alt,2,3,True,10)
    st(f"  {alt} m AGL", s['error_m'], 20)

print("\nST-4: Low frame counts")
for n in [1, 3, 5, 10]:
    s = run_scenario(f"n={n}", BASE_LAT,BASE_LON,45,80,2,3,True,n)
    limit = 30 if n == 1 else 20
    st(f"  {n} frame(s)  (limit={limit}m)", s['error_m'], limit)

print("\nST-5: Bad GPS (6 m CEP), 10 frames")
s = run_scenario("badGPS", BASE_LAT,BASE_LON,45,80,2,6,True,10)
st("  6 m CEP, 10 frames", s['error_m'], 25)

print("\nST-6: Confidence threshold rejects injected false detections")
flt_cf = simulate_flight(FLAG_LAT, FLAG_LON, 80, 45, 18, 20, 30, 3.)
det_cf = simulate_detections(FLAG_LAT, FLAG_LON, flt_cf, CAM)
# Inject 10 corner-of-frame false alarms with low confidence
for _ in range(10):
    det_cf.append({'u':10,'v':10,'altitude':80,'lat_uav':LAT_REF,
                   'lon_uav':31.25,'roll':0,'pitch':0,'yaw':45,'confidence':0.15})
r_cf = localize_flag(det_cf, CAM, min_confidence=0.4)
err_cf = haversine_m(*r_cf, FLAG_LAT, FLAG_LON) if r_cf else None
st("  10 false alarms injected (filtered by conf)", err_cf, 20)

print("\nST-7: Geometry guard rejects extreme angles")
r_guard = localize(CAM.cx, CAM.cy, CAM, 80, LAT_REF, 31.25, pitch_deg=89)
ok = r_guard is None
if not ok: PASS_ALL = False
print(f"  {'✅' if ok else '❌'} pitch=89° → returns None  (got: {r_guard})")

print("\nST-8: Distortion correction makes a measurable difference at edge of frame")
# Edge pixel (near corner of frame)
u_edge, v_edge = 50., 50.
u_raw_und, v_raw_und = undistort_pixel(u_edge, v_edge, CAM.K, CAM.dist)
pixel_shift = math.sqrt((u_raw_und-u_edge)**2 + (v_raw_und-v_edge)**2)
ok = pixel_shift > 0.01   # should be non-trivial with real distortion coefficients
if not ok: PASS_ALL = False
print(f"  {'✅' if ok else '❌'} Corner undistortion shift = {pixel_shift:.3f} px  "
      f"(→ ~{pixel_shift*80/CAM.fx:.3f} m ground error if ignored)")

print()
print("="*55)
print("ALL STRESS TESTS PASSED ✅" if PASS_ALL else "SOME STRESS TESTS FAILED ❌ — see above")


## Section 9 — Swap-In Guide for Real Data

When you have a real flight, make these four changes. Everything else is untouched.

---
### 1. Camera — load `camera_params.yaml`

```python
import yaml
with open('camera_params.yaml') as f: p = yaml.safe_load(f)
CAM = CameraIntrinsics(fx=p['fx'], fy=p['fy'], cx=p['cx'], cy=p['cy'],
                        dist=np.array(p['dist_coeffs']))
```

---
### 2. Telemetry — parse ArduPilot `.bin`

```python
from pymavlink import mavutil
mlog = mavutil.mavlink_connection('flight.bin')
frames = []
gps, att, baro = {}, {}, {}
while True:
    msg = mlog.recv_match(blocking=False)
    if msg is None: break
    t = msg._timestamp
    if msg.get_type()=='GPS':  gps  = {'lat':msg.Lat/1e7,'lon':msg.Lng/1e7,'t':t}
    if msg.get_type()=='ATT':  att  = {'roll':msg.Roll,'pitch':msg.Pitch,
                                        'yaw':msg.Yaw+4.0,'t':t}  # +4° Cairo declination
    if msg.get_type()=='BARO': baro = {'alt':msg.Alt,'t':t}
    if gps and att and baro:
        frames.append(TelemetryFrame(t, gps['lat'], gps['lon'], baro['alt'],
                                      att['roll'], att['pitch'], att['yaw']))
FLIGHT = frames
```

---
### 3. Detections — run YOLO on video

```python
from ultralytics import YOLO; import cv2
model = YOLO('flag_detector.pt'); cap = cv2.VideoCapture('flight.MP4')
fps_v = cap.get(cv2.CAP_PROP_FPS); dets = []; fi = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    if fi % 5 == 0:   # sample every 5th frame
        for box in model(frame, verbose=False)[0].boxes:
            t = fi / fps_v
            tel = min(FLIGHT, key=lambda f: abs(f.time_s - t))
            dets.append({'u':(float(box.xyxy[0][0])+float(box.xyxy[0][2]))/2,
                          'v':(float(box.xyxy[0][1])+float(box.xyxy[0][3]))/2,
                          'confidence':float(box.conf),
                          'altitude':tel.alt_agl,'lat_uav':tel.lat_uav,
                          'lon_uav':tel.lon_uav,'roll':tel.roll_deg,
                          'pitch':tel.pitch_deg,'yaw':tel.yaw_deg})
    fi += 1
DETECTIONS = dets
```

---
### 4. Ground truth — read from CSV

```python
import csv
FLAGS = {}
with open('surveyed_flags.csv') as f:
    for row in csv.DictReader(f):
        FLAGS[row['name']] = (float(row['lat']), float(row['lon']))
```

---
### Processing time estimate (15-min video, 1080p30)

| Frame step | Frames | CPU time | GPU time |
|---|---|---|---|
| Every frame | 27,000 | ~45 min | ~5 min |
| Every 5th (recommended) | 5,400 | ~9 min | ~1 min |
| Every 10th | 2,700 | ~4 min | ~30 s |

Recommendation: `frame_step=5`. A 3-second detection window at 30 FPS = 18 detections → 6 after sampling → well above the 5-frame accuracy floor.
